In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/cleaned_smartbiz_dataset.csv")
df.head()

,Units Sold,Unit Price,Total Revenue,Rating,Reviews,year,month,day,Product Category_Books,Product Category_Clothing,Product Category_Electronics,Product Category_Home Appliances,Product Category_Sports,Region_Europe,Region_North America,Payment Method_Debit Card,Payment Method_PayPal
0,2,999.99,1999.98,1,I always wanted to have an I phone i got it no...,2024,11,1,False,False,True,False,False,False,True,False,False
1,1,499.99,499.99,0,A terrible experience. The vacuum cleaner does...,2024,11,1,False,False,False,True,False,True,False,False,True
2,3,69.99,209.97,1,I regret buying this. It's of very low quality...,2024,11,1,False,True,False,False,False,False,False,True,False
3,1,89.99,89.99,4,Absolutely fantastic after one month of use! D...,2024,11,1,False,False,False,False,False,True,False,False,True
4,2,180.00,360.00,1,The quality is shocking. Avoid this product.,2024,11,1,False,True,False,False,False,False,False,False,False


In [3]:
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

In [4]:
df['Reviews'].fillna(df['Reviews'].median(), inplace=True)

/tmp/ipykernel_18746/1410591978.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Reviews'].fillna(df['Reviews'].median(), inplace=True)


In [5]:
df['Revenue_per_Unit'] = df['Total Revenue'] / df['Units Sold']

In [6]:
df['Sales_Efficiency'] = df['Rating'] * df['Units Sold']

In [7]:
df['Price_Category'] = pd.cut(df['Unit Price'],
                             bins=[0, 100, 500, 2000],
                             labels=['Low', 'Medium', 'High'])

In [8]:
df = pd.get_dummies(df, columns=['Price_Category'], drop_first=True)

In [9]:
df['Is_Weekend'] = df['day'].apply(lambda x: 1 if x in [6,7] else 0)

In [10]:
import numpy as np

In [11]:
df['Month_Sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [12]:
bool_cols = df.select_dtypes(include='bool').columns

for col in bool_cols:
    df[col] = df[col].astype(int)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = df.select_dtypes(include=['int64','float64']).columns

df[num_cols] = scaler.fit_transform(df[num_cols])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [39]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [43]:
df['Target'] = (df['Units Sold'] > df['Units Sold'].median()).astype(int)

In [44]:
X = df.drop(columns=['Target', 'Units Sold'])
y = df['Target']

In [45]:
X = X.fillna(X.median())

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [48]:
knn = KNeighborsClassifier(n_neighbors=5)

# Drop 'Reviews' as it contains only NaNs from the previous numeric conversion
X_train_clean = X_train.drop(columns=['Reviews'], errors='ignore')
X_test_clean = X_test.drop(columns=['Reviews'], errors='ignore')

knn.fit(X_train_clean, y_train)
knn_acc = accuracy_score(y_test, knn.predict(X_test_clean))
print(f'KNN Accuracy: {knn_acc}')

KNN Accuracy: 0.8631578947368421


In [49]:
print(f"KNN Accuracy: {knn_acc:.2%}")

KNN Accuracy: 86.32%
